# 04 - Causal Analysis
## Estimate incremental sales, separate promotion impact from natural sales changes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

In [ ]:
DATA_DIR = '../data/raw/'

transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')
campaign_desc = pd.read_csv(f'{DATA_DIR}campaign_desc.csv')
campaign_table = pd.read_csv(f'{DATA_DIR}campaign_table.csv')

daily = transactions.groupby('DAY', as_index=False).agg(
    sales=('SALES_VALUE', 'sum'),
    quantity=('QUANTITY', 'sum'),
    discount=('RETAIL_DISC', lambda x: abs(x).sum()),
)

print(f'Total days of data: {len(daily)}')
print(f'Date range: {daily["DAY"].min()} to {daily["DAY"].max()}')

In [ ]:
# Difference-in-Differences analysis for a single campaign
def estimate_campaign_impact(campaign_id):
    camp = campaign_desc[campaign_desc['CAMPAIGN'] == campaign_id]
    if camp.empty:
        return None
    
    start = int(camp['START_DAY'].iloc[0])
    end = int(camp['END_DAY'].iloc[0])
    
    pre = daily[(daily['DAY'] >= start - 28) & (daily['DAY'] < start)]
    during = daily[(daily['DAY'] >= start) & (daily['DAY'] <= end)]
    post = daily[(daily['DAY'] > end) & (daily['DAY'] <= end + 28)]
    
    pre_sales = pre['sales'].sum()
    during_sales = during['sales'].sum()
    post_sales = post['sales'].sum() if not post.empty else 0
    
    during_days = max(1, end - start + 1)
    daily_baseline = pre_sales / 28.0
    expected_sales = daily_baseline * during_days
    incremental_sales = during_sales - expected_sales
    
    pre_discount = pre['discount'].sum()
    during_discount = during['discount'].sum()
    promo_cost = during_discount
    
    avg_price = during['sales'].sum() / during['quantity'].sum() if during['quantity'].sum() > 0 else 0
    incremental_revenue = incremental_sales
    incremental_profit = incremental_revenue - promo_cost
    roi = incremental_profit / promo_cost if promo_cost > 0 else 0.0
    
    # Pre-trend adjustment
    pre_trend = 0
    if len(pre) > 3:
        pre_daily = pre.groupby('DAY')['sales'].sum().reset_index()
        if len(pre_daily) > 3:
            x = np.arange(len(pre_daily)).reshape(-1, 1)
            y = pre_daily['sales'].values
            lr = LinearRegression()
            lr.fit(x, y)
            pre_trend = float(lr.coef_[0] * during_days)
    
    # Seasonality adjustment
    seasonality_effect = 0
    if not post.empty:
        post_days = post['DAY'].nunique()
        if post_days > 0:
            post_daily_avg = post_sales / post_days
            seasonality_effect = (post_daily_avg - daily_baseline) * during_days
    
    adjusted_incremental = incremental_sales - pre_trend - seasonality_effect
    
    return {
        'campaign_id': campaign_id,
        'start_day': start,
        'end_day': end,
        'pre_sales': round(pre_sales, 2),
        'during_sales': round(during_sales, 2),
        'post_sales': round(post_sales, 2),
        'expected_sales': round(expected_sales, 2),
        'incremental_sales_raw': round(incremental_sales, 2),
        'incremental_sales_adjusted': round(adjusted_incremental, 2),
        'promotion_cost': round(promo_cost, 2),
        'incremental_revenue': round(incremental_revenue, 2),
        'incremental_profit': round(incremental_profit, 2),
        'roi': round(roi, 4),
        'pre_trend_effect': round(pre_trend, 2),
        'seasonality_effect': round(seasonality_effect, 2),
    }

In [ ]:
impact = estimate_campaign_impact(1)
impact

In [ ]:
# Analyze all campaigns
all_impacts = []
for cid in campaign_desc['CAMPAIGN'].unique():
    result = estimate_campaign_impact(cid)
    if result:
        all_impacts.append(result)

impact_df = pd.DataFrame(all_impacts)
impact_df = impact_df.sort_values('roi', ascending=False)
print(f'Analyzed {len(impact_df)} campaigns')
impact_df[['campaign_id', 'roi', 'incremental_sales_raw', 'incremental_sales_adjusted', 'promotion_cost']].head(10)

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.bar(range(len(impact_df)), impact_df['roi'])
plt.axhline(y=0, color='red', linestyle='--')
plt.title('ROI by Campaign')
plt.xlabel('Campaign Index')
plt.ylabel('ROI')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(impact_df['promotion_cost'], impact_df['incremental_sales_adjusted'], alpha=0.6, s=80)
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Incremental Sales vs Promotion Cost')
plt.xlabel('Promotion Cost ($)')
plt.ylabel('Adjusted Incremental Sales ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize a specific campaign
def plot_campaign_timeline(campaign_id):
    camp = campaign_desc[campaign_desc['CAMPAIGN'] == campaign_id]
    if camp.empty:
        return
    start, end = int(camp['START_DAY'].iloc[0]), int(camp['END_DAY'].iloc[0])
    
    window = daily[(daily['DAY'] >= start - 30) & (daily['DAY'] <= end + 30)]
    
    plt.figure(figsize=(14, 5))
    plt.plot(window['DAY'], window['sales'], marker='.', linewidth=1.5, label='Daily Sales')
    plt.axvline(x=start, color='green', linestyle='--', linewidth=2, label='Campaign Start')
    plt.axvline(x=end, color='red', linestyle='--', linewidth=2, label='Campaign End')
    
    pre_sales = window[window['DAY'] < start]['sales'].mean()
    plt.axhline(y=pre_sales, color='gray', linestyle=':', alpha=0.7, label='Pre-period avg')
    
    plt.title(f'Campaign {campaign_id}: Sales Timeline')
    plt.xlabel('Day')
    plt.ylabel('Sales ($)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_campaign_timeline(1)

In [ ]:
# Summary statistics
print('=== Causal Analysis Summary ===')
print(f'Total campaigns analyzed: {len(impact_df)}')
print(f'Positive ROI campaigns: {(impact_df["roi"] > 0).sum()}')
print(f'Negative ROI campaigns: {(impact_df["roi"] <= 0).sum()}')
print(f'\nROI stats:')
print(impact_df['roi'].describe())
print(f'\nIncremental sales stats:')
print(impact_df['incremental_sales_adjusted'].describe())